# Regressione delta_tas vs delta_cover (adattata)

Disegno **non circolare**: SENS e' forzato con osservazioni di vegetation cover
(cvh/cvl), quindi `cover_SENS ~ obs`. La quantita' `delta_cover = cover_SENS -
cover_CTRL` rappresenta gia' la correzione rispetto al modello di vegetazione
dinamico (libero) di CTRL, senza bisogno di un dataset osservativo esterno.

Regredisce, per ciascuna delle 10 combinazioni di lead year (annuale):

- `delta_tas(anno) = anomalia_tas_SENS(anno) - anomalia_tas_CTRL(anno)`
- `delta_cover(anno) = anomalia_cover_SENS(anno) - anomalia_cover_CTRL(anno)`

sugli anni in comune tra le due serie (allineamento per anno solare — la
cover e' una serie continua 1993-2019, tas e' un hindcast DCPP con un valore
per anno di inizializzazione).

Output per ciascuna combinazione lead-year x {cvh, cvl}:
- mappa slope/p-value per pixel (`af.map_plot`)
- regressione scalare + scatter sul box Siberia (54-70N, 88-110E), stesso box
  di `07-covariance_regression.ipynb` / `08-covariance_plot.ipynb`


In [ ]:
# rende config.py (in notebooks/) importabile anche da questa sottocartella
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025

exp_ctrl = 'a1ua'
exp_sens = 'a52o'
variables = ['cvh', 'cvl']
SAVE_PATH = str(FIG_DIR)


In [ ]:
# La logica di calcolo sta in un modulo importabile (cover_tas_lib.py): i
# processi spawn usati sotto (ogni task in un processo fresco, per evitare
# l'accumulo che ha gia' causato crash del pool nel calcolo del bias stagionale
# in questa stessa sessione) non vedono le funzioni definite nel notebook.
sys.path.insert(0, os.getcwd())
from cover_tas_lib import run_one_adapted, LEADS


In [ ]:
%%time
import multiprocessing as mp

jobs = [(exp_ctrl, exp_sens, var, y1, y2, SAVE_PATH) for var in variables for (y1, y2) in LEADS]

with mp.get_context('spawn').Pool(processes=2, maxtasksperchild=1) as pool:
    for _i, msg in enumerate(pool.imap_unordered(run_one_adapted, jobs), 1):
        print(f"  {msg}   {_i}/{len(jobs)}", flush=True)
